# 06_01 - Decision Engine: Heuristic Policy
## 1. Methodology Overview

This notebook builds the interpretable rule-based decision engine that consumes quantile forecasts and market inputs.
It follows the same project modules used by the pipeline so results are reproducible and consistent with backtesting.

**Source data used in this notebook:**
- `data/processed/train_features.csv`
- `data/processed/validation_features.csv`

**Core modules used in this notebook:**
- `src/models/train_model.py`
- `src/decision/policy_inputs.py`
- `src/decision/heuristic_policy.py`

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = Path('../../').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

train_df = pd.read_csv(project_root / 'data/processed/train_features.csv')
validation_df = pd.read_csv(project_root / 'data/processed/validation_features.csv')

print(f'Train features: {train_df.shape[0]} rows x {train_df.shape[1]} columns')
print(f'Validation features: {validation_df.shape[0]} rows x {validation_df.shape[1]} columns')
display(validation_df.head())

## 2. Build Policy Inputs and Apply the Heuristic Engine

We train the quantile suite, transform outputs into policy-ready inputs, and then apply the heuristic rules.
This mirrors the production flow used in the pipeline.

In [ ]:
from src.models.train_model import train_quantile_suite
from src.decision.policy_inputs import prepare_policy_inputs, summarize_policy_inputs
from src.decision.heuristic_policy import apply_heuristic_policy, summarize_policy_actions

quantile_output = train_quantile_suite(train_df=train_df, eval_df=validation_df)
policy_inputs_df = prepare_policy_inputs(validation_df, quantile_output.results)
heuristic_policy_df = apply_heuristic_policy(policy_inputs_df)

print('Policy input overview:')
display(summarize_policy_inputs(policy_inputs_df))

print('Policy output preview:')
display(
    heuristic_policy_df[[
        'date',
        'forecast_central',
        'forecast_tail',
        'current_m1_future',
        'tail_vs_future_abs',
        'tail_vs_central_abs',
        'recommended_action',
        'decision_reason',
    ]].head(12)
)

## 3. Action Mix and Risk-Context Diagnostics

This section summarizes how often each action is recommended and shows context statistics by action.

In [ ]:
action_summary_df = summarize_policy_actions(heuristic_policy_df)
display(action_summary_df)

context_summary = (
    heuristic_policy_df.groupby('recommended_action')[['tail_vs_future_abs', 'tail_vs_central_abs']]
    .agg(['mean', 'median', 'max'])
    .round(3)
)
display(context_summary)